<a href="https://colab.research.google.com/github/replyeshab/CineAI-AI-Based-Hybrid-Recommendation-System/blob/main/hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Recommendation

In [8]:
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import MinMaxScaler

In [9]:
import joblib

ARTIFACT_PATH = "/content/drive/MyDrive/CineAI/Artifact"

movie_data = joblib.load(ARTIFACT_PATH+"/movie_data.pkl")

popular_movies = joblib.load(ARTIFACT_PATH+"/popular_movies.pkl")

svd_model = joblib.load(ARTIFACT_PATH+"/svd_model.pkl")

knn_model = joblib.load(ARTIFACT_PATH+"/knn_content_model.pkl")

tfidf_matrix = joblib.load(ARTIFACT_PATH+"/tfidf_matrix.pkl")

tfidf_vectorizer = joblib.load(ARTIFACT_PATH+"/tfidf_vectorizer.pkl")

title_lookup = joblib.load(ARTIFACT_PATH+"/title_lookup.pkl")

title_to_index = joblib.load(ARTIFACT_PATH+"/title_to_index.pkl")

user_encoder = joblib.load(ARTIFACT_PATH+"/user_encoder.pkl")

movie_encoder = joblib.load(ARTIFACT_PATH+"/movie_encoder.pkl")

user_features = joblib.load(ARTIFACT_PATH + "/user_features.pkl")
movie_features = joblib.load(ARTIFACT_PATH + "/movie_features.pkl")


In [10]:
ratings = pd.read_csv(
    "/content/drive/MyDrive/CineAI/ml-32m/ml-32m/ratings.csv"
)

In [11]:
movie_stats = pd.read_pickle("/content/drive/MyDrive/CineAI/Artifact/movie_stats.pkl")

# Popularity Recommendation

In [12]:
def calculate_imdb_score(df, quantile=0.90):
    """
    Calculates IMDb Weighted Rating.

    Parameters
    ----------
    df : DataFrame
    quantile : float
        Minimum vote percentile.

    Returns
    -------
    DataFrame
    """

    df = df.copy()

    C = df["average_rating"].mean()

    m = df["rating_count"].quantile(quantile)

    df["weighted_rating"] = (
        (
            df["rating_count"] /
            (df["rating_count"] + m)
        ) * df["average_rating"]
    ) + (
        (
            m /
            (df["rating_count"] + m)
        ) * C
    )

    return df

In [13]:
popular_movies = calculate_imdb_score(movie_stats)

In [14]:
popular_movies = (
    popular_movies
    .sort_values(
        by="weighted_rating",
        ascending=False
    )
    .reset_index(drop=True)
)

In [15]:
def recommend_popular(top_n=500):

    columns = [
        "movieId",
        "clean_title",
        "genres",
        "average_rating",
        "rating_count",
        "weighted_rating"
    ]

    return popular_movies[columns].head(top_n)

In [16]:
popular = recommend_popular(20)
popular.head()

,movieId,clean_title,genres,average_rating,rating_count,weighted_rating
0,318,"Shawshank Redemption, The",Crime|Drama,4.404614,102929,4.401238
1,159817,Planet Earth,Documentary,4.444369,2948,4.332311
2,858,"Godfather, The",Crime|Drama,4.317030,66440,4.312134
3,170705,Band of Brothers,Action|Drama|War,4.426539,2811,4.310914
4,202439,Parasite,Comedy|Drama,4.312254,11670,4.284956


# Cold Start

In [17]:
def recommend_cold_start(popular_df, top_n=20):
    """
    Recommend movies for brand-new users.

    Uses popularity only because the user has no history.
    """

    cold_start = (
        popular_df
        .sort_values(
            by=[
                "weighted_rating",
                "rating_count",
                "average_rating"
            ],
            ascending=False
        )
        .head(top_n)
    )

    return cold_start

In [18]:
USER_ID = 1
def is_new_user(user_id, ratings):

    if user_id is None:
        return True

    if user_id not in ratings["userId"].values:
        return True

    return False

def is_sparse_user(
    user_id,
    ratings,
    threshold=10
):

    if is_new_user(user_id, ratings):
        return False

    n = len(
        ratings[
            ratings["userId"] == user_id
        ]
    )

    return n < threshold

In [19]:
if is_new_user(USER_ID, ratings):

    print("Cold Start User")

    final_recommendations = recommend_cold_start(
        popular_movies,
        top_n=500
    )

    display(final_recommendations)

    raise SystemExit()

# Collaborative Recommendation

In [20]:
def rank_candidates(
    candidate_indices,
    scores,
    top_n=500
):

    top_indices = candidate_indices[:top_n]

    movie_ids = movie_encoder.inverse_transform(
        top_indices
    )

    recommendations = (
        movie_data[
            movie_data.movieId.isin(
                movie_ids
            )
        ]
        .copy()
    )

    score_map = {
        movie_encoder.inverse_transform([idx])[0]:
        scores[idx]
        for idx in top_indices
    }

    recommendations["svd_score"] = (
        recommendations.movieId
        .map(score_map)
    )

    return recommendations.sort_values(
        "svd_score",
        ascending=False
    )

In [21]:
def filter_seen_movies(
    user_id,
    candidate_indices,
    scores
):

    watched = ratings.loc[
        ratings.userId == user_id,
        "movieId"
    ]

    watched = watched[
        watched.isin(
            movie_encoder.classes_
        )
    ]

    watched_indices = set(
        movie_encoder.transform(
            watched
        )
    )

    filtered = [
        idx
        for idx in candidate_indices
        if idx not in watched_indices
    ]

    return filtered

In [22]:
def generate_candidates(
    user_id,
    candidate_size=500
):

    if user_id not in user_encoder.classes_:
        raise ValueError("Unknown User ID")

    user_index = user_encoder.transform(
        [user_id]
    )[0]

    user_vector = user_features[user_index]

    scores = movie_features @ user_vector

    top_candidates = np.argsort(
        scores
    )[::-1][:candidate_size]

    return top_candidates, scores

In [23]:
def recommend_for_user_collaborative(
    user_id,
    top_n=500
):

    candidates, scores = generate_candidates(
        user_id
    )

    filtered_candidates = filter_seen_movies(
        user_id,
        candidates,
        scores
    )

    recommendations = rank_candidates(
        filtered_candidates,
        scores,
        top_n
    )

    recommendations = (
        recommendations
        .sort_values("svd_score", ascending=False)
        .reset_index(drop=True)
    )

    recommendations.index.name = "Rank"

    return recommendations

In [24]:
collab = recommend_for_user_collaborative(user_id=1, top_n=20)
collab.head()

,movieId,title,genres,year,clean_title,genres_list,imdbId,tmdbId,tag,combined_features,svd_score
Rank,,,,,,,,,,,
0,858,"Godfather, The (1972)",Crime|Drama,1972.0,"Godfather, The","[Crime, Drama]",68646,238.0,al pacino atmospheric great acting masterpiece...,"Godfather, The Crime Drama Al Pacino atmospher...",2.863783
1,1222,Full Metal Jacket (1987),Drama|War,1987.0,Full Metal Jacket,"[Drama, War]",93058,600.0,vietnam war tumey s dvds imdb top 250 stanley ...,Full Metal Jacket Drama War Vietnam War Tumey'...,2.771993
2,924,2001: A Space Odyssey (1968),Adventure|Drama|Sci-Fi,1968.0,2001: A Space Odyssey,"[Adventure, Drama, Sci-Fi]",62622,62.0,atmospheric space stanley kubrick ambiguous at...,2001: A Space Odyssey Adventure Drama Sci-Fi a...,2.545833
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,1964.0,Dr. Strangelove or: How I Learned to Stop Worr...,"[Comedy, War]",57012,935.0,black and white stanley kubrick classic dark c...,Dr. Strangelove or: How I Learned to Stop Worr...,2.479119
4,2858,American Beauty (1999),Drama|Romance,1999.0,American Beauty,"[Drama, Romance]",169547,14.0,bittersweet great acting kevin spacey loneline...,American Beauty Drama Romance bittersweet grea...,2.291048


# Content Recommedation

In [25]:
from sklearn.metrics.pairwise import linear_kernel
def recommend_content(title, top_n=500):

    matches = movie_data[
        movie_data["clean_title"]
        .str.lower() == title.lower()
    ]

    if matches.empty:
        print("Movie not found.")
        return pd.DataFrame(
          columns=[
             "movieId",
            "similarity_score"
    ]
)

    idx = matches.index[0]

    cosine_scores = linear_kernel(
        tfidf_matrix[idx:idx + 1],
        tfidf_matrix
    ).flatten()

    similar_indices = np.argsort(cosine_scores)[::-1]

    similar_indices = similar_indices[1:top_n + 1]

    recommendations = movie_data.iloc[
        similar_indices
    ][["movieId", "clean_title", "genres", "year"]].copy()

    recommendations["similarity_score"] = (
        cosine_scores[similar_indices]
    )

    return recommendations

In [26]:
def generate_candidate_pool(
    user_id,
    reference_movie,
    top_n=500
):
  collab = recommend_for_user_collaborative(
    user_id=user_id,
    top_n=top_n
)
  content = recommend_content(
    reference_movie,
    top_n
)
  popular = recommend_popular(
    top_n
)
  collab = (
    collab[
        [
            "movieId",
            "svd_score"
        ]
    ]
    .rename(
        columns={
            "svd_score":"collab_score"
        }
    )
)
  content = (
    content[
        [
            "movieId",
            "similarity_score"
        ]
    ]
    .rename(
        columns={
            "similarity_score":"content_score"
        }
    )
)
  popular = (
    popular[
        [
            "movieId",
            "average_rating",
            "rating_count",
            "weighted_rating"
        ]
    ]
    .rename(
        columns={
            "weighted_rating":"popularity_score"
        }
    )
)
  hybrid = (
    collab
    .merge(
        content,
        on="movieId",
        how="outer"
    )
    .merge(
        popular,
        on="movieId",
        how="outer"
    )
)
  score_columns = [
    "collab_score",
    "content_score",
    "popularity_score"
]

  for col in score_columns:

    hybrid[col] = scaler.fit_transform(
        hybrid[[col]].fillna(0)
    )
  return hybrid

In [27]:
content = recommend_content("Toy Story", 20)
content.head()

,movieId,clean_title,genres,year,similarity_score
3021,3114,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.892434
2264,2355,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.806176
14815,78499,Toy Story 3,Adventure|Animation|Children|Comedy|Fantasy|IMAX,2010.0,0.711669
39850,157296,Finding Dory,Adventure|Animation|Comedy,2016.0,0.701200
4781,4886,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,2001.0,0.700583


In [28]:
print(collab.columns)
print(content.columns)
print(popular.columns)

Index(['movieId', 'title', 'genres', 'year', 'clean_title', 'genres_list',
       'imdbId', 'tmdbId', 'tag', 'combined_features', 'svd_score'],
      dtype='object')
Index(['movieId', 'clean_title', 'genres', 'year', 'similarity_score'], dtype='object')
Index(['movieId', 'clean_title', 'genres', 'average_rating', 'rating_count',
       'weighted_rating'],
      dtype='object')


In [29]:
print(collab.head())
print(content.head())
print(popular.head())

      movieId                                              title  \
Rank                                                               
0         858                              Godfather, The (1972)   
1        1222                           Full Metal Jacket (1987)   
2         924                       2001: A Space Odyssey (1968)   
3         750  Dr. Strangelove or: How I Learned to Stop Worr...   
4        2858                             American Beauty (1999)   

                      genres    year  \
Rank                                   
0                Crime|Drama  1972.0   
1                  Drama|War  1987.0   
2     Adventure|Drama|Sci-Fi  1968.0   
3                 Comedy|War  1964.0   
4              Drama|Romance  1999.0   

                                            clean_title  \
Rank                                                      
0                                        Godfather, The   
1                                     Full Metal Jacket   
2     

# Genre Recommedation

In [30]:
def build_genre_profile(
    user_id,
    ratings,
    movie_data
):
    """
    Build a weighted genre preference profile
    for one user.

    Higher rated movies contribute more
    than lower rated movies.
    """

    user_ratings = ratings[
        ratings["userId"] == user_id
    ]

    profile = {}

    user_movies = user_ratings.merge(
        movie_data[
            [
                "movieId",
                "genres"
            ]
        ],
        on="movieId",
        how="left"
    )

    for _, row in user_movies.iterrows():

        if pd.isna(row["genres"]):
            continue

        genres = row["genres"].split("|")

        rating = row["rating"]

        for genre in genres:

            if genre == "(no genres listed)":
                continue

            profile[genre] = (
                profile.get(genre, 0)
                + rating
            )

    if len(profile) == 0:
        return {}

    maximum = max(profile.values())

    for genre in profile:

        profile[genre] /= maximum

    return profile

In [31]:
def calculate_genre_score(hybrid_df, genre_profile):

    genre_scores = []

    for _, row in hybrid_df.iterrows():

        if pd.isna(row["genres"]):
            genre_scores.append(0)
            continue

        movie_genres = row["genres"].split("|")

        score = 0

        count = 0

        for genre in movie_genres:

            if genre == "(no genres listed)":
                continue

            score += genre_profile.get(genre, 0)

            count += 1

        if count == 0:
            genre_scores.append(0)

        else:
            genre_scores.append(score / count)

    hybrid_df["genre_score"] = genre_scores

    return hybrid_df

In [32]:
import datetime

def build_features(
    df,
    user_id
):
  df["rating_score"] = scaler.fit_transform(
    df[
        ["average_rating"]
    ].fillna(0)
)
  df["rating_count_score"] = scaler.fit_transform(
    df[
        ["rating_count"]
    ].fillna(0)
)
  df = df.merge(
    movie_data[
        ["movieId", "year","genres"]
    ],
    on="movieId",
    how="left"
)
  watched_movies = ratings.loc[
    ratings["userId"] == user_id,
    "movieId"
]

  df = df[
    ~df["movieId"].isin(
        watched_movies
    )
]
  genre_profile = build_genre_profile(
    user_id,
    ratings,
    movie_data
)

  df = calculate_genre_score(
    df,
    genre_profile
)
  df["genre_score"] = scaler.fit_transform(
    df[
        ["genre_score"]
    ].fillna(0)
)
  new_movie_mask = (
    df["rating_count"].fillna(0) == 0
)

  df.loc[
    new_movie_mask,
    "collab_score"
] = np.nan

  df.loc[
    new_movie_mask,
    "popularity_score"
] = np.nan

  CURRENT_YEAR = datetime.datetime.now().year

  df["movie_age"] = CURRENT_YEAR - df["year"]

  df["recency_score"] = 1 / (1 + df["movie_age"])

  df["recency_score"] = scaler.fit_transform(
    df[["recency_score"]]
)
  return df

In [33]:
def get_dynamic_weights(
    user_id,
    ratings
):

    if is_new_user(
        user_id,
        ratings
    ):

        return {
            "collab_score":0,
            "content_score":0,
            "popularity_score":0.45,
            "genre_score":0,
            "rating_score":0.25,
            "rating_count_score":0.20,
            "recency_score":0.10
        }

    if is_sparse_user(
        user_id,
        ratings
    ):

        return {
            "collab_score":0.10,
            "content_score":0.40,
            "popularity_score":0.15,
            "genre_score":0.25,
            "rating_score":0.05,
            "rating_count_score":0.03,
            "recency_score":0.02
        }

    return {
        "collab_score":0.20,
        "content_score":0.30,
        "popularity_score":0.15,
        "genre_score":0.20,
        "rating_score":0.05,
        "rating_count_score":0.05,
        "recency_score":0.05
    }

In [34]:
score_columns = [
    "collab_score",
    "content_score",
    "popularity_score",
    "genre_score",
    "rating_score",
    "rating_count_score",
    "movie_age"
]

In [35]:
def rank_movies(
    df,
    user_id,
    ratings
):

    weights = get_dynamic_weights(
        user_id,
        ratings
    )

    final_scores = []

    for _, row in df.iterrows():

        score = 0
        total_weight = 0

        for feature, weight in weights.items():

            value = row[feature]

            if pd.notna(value):

                score += value * weight

                total_weight += weight

        if total_weight > 0:

            score /= total_weight

        final_scores.append(score)

    df["final_score"] = final_scores

    return df

In [36]:
def validate_recommendations(
    recommendations,
    watched_movies
):

    assert (
        recommendations["movieId"]
        .duplicated()
        .sum() == 0
    ), "Duplicate recommendations"

    assert (
        len(
            set(
                recommendations["movieId"]
            ).intersection(
                watched_movies
            )
        ) == 0
    ), "Already watched movie recommended"

    assert (
        recommendations["final_score"]
        .isna()
        .sum() == 0
    ), "NaN score detected"

    print("All validation checks passed.")

In [37]:
def get_final_recommendations(
    df,
    top_n=500
):
    df = df.sort_values(
        "final_score",
        ascending=False
    )

    return df.head(top_n)

In [38]:
def recommend_movies(
    user_id,
    reference_movie,
    top_n=500
):


    if is_new_user(user_id, ratings):

        print("New User Detected")

        return recommend_cold_start(
            popular_movies,
            top_n=top_n
        )


    candidate_pool = generate_candidate_pool(
        user_id=user_id,
        reference_movie=reference_movie,
        top_n=500
    )
    if candidate_pool.empty:
      print("Candidate pool empty. Falling back to popularity.")
      return recommend_popular(top_n)

    candidate_pool = build_features(
        df=candidate_pool,
        user_id=user_id
    )



    candidate_pool = rank_movies(
        df=candidate_pool,
        user_id=user_id,
        ratings=ratings
    )



    recommendations = get_final_recommendations(
        candidate_pool,
        top_n=top_n
    )



    watched_movies = ratings.loc[
        ratings["userId"] == user_id,
        "movieId"
    ]

    validate_recommendations(
        recommendations,
        watched_movies
    )

    return recommendations

In [39]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

final_recommendations = recommend_movies(
    user_id=1,
    reference_movie="Toy Story",
    top_n=20
)

display(final_recommendations)

All validation checks passed.


,movieId,collab_score,content_score,average_rating,rating_count,popularity_score,rating_score,rating_count_score,year,genres,genre_score,movie_age,recency_score,final_score
94,858,1.000000,0.000000,4.317030,66440.0,0.979755,0.970811,0.645493,1972.0,Crime|Drama,0.585635,54.0,0.044414,0.547126
173,1193,0.546209,0.000000,4.204229,44592.0,0.953726,0.945444,0.433231,1975.0,Drama,1.000000,51.0,0.048738,0.523671
183,1207,0.598914,0.000000,4.134469,18785.0,0.936032,0.929756,0.182504,1962.0,Drama,1.000000,64.0,0.032883,0.517445
196,1222,0.967948,0.000000,4.049029,32501.0,0.918172,0.910543,0.315761,1987.0,Drama|War,0.600829,39.0,0.072519,0.516422
443,2858,0.800008,0.000000,4.093051,65158.0,0.929037,0.920442,0.633038,1999.0,Drama|Romance,0.661602,27.0,0.116685,0.515186
484,3114,NaN,1.000000,NaN,NaN,NaN,0.000000,0.000000,1999.0,Adventure|Animation|Children|Comedy|Fantasy,0.141436,27.0,0.116685,0.514033
253,1293,0.524371,0.000000,3.979554,11078.0,0.899324,0.894919,0.107628,1982.0,Drama,1.000000,44.0,0.061069,0.492954
264,1358,0.462583,0.000000,3.998711,14354.0,0.904694,0.899227,0.139455,1996.0,Drama,1.000000,30.0,0.102438,0.485277
389,2355,NaN,0.903345,NaN,NaN,NaN,0.000000,0.000000,1998.0,Adventure|Animation|Children|Comedy,0.162983,28.0,0.111608,0.475663
215,1246,0.367304,0.000000,3.938424,31035.0,0.893158,0.885670,0.301519,1989.0,Drama,1.000000,37.0,0.077943,0.470691


# Evaluation Matrix

In [40]:
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split

In [41]:
from sklearn.model_selection import train_test_split

def split_ratings_per_user(
    ratings,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=42
):

    train_list = []
    val_list = []
    test_list = []

    for user_id, user_data in ratings.groupby("userId"):

        # Handle sparse users
        if len(user_data) <= 2:
            train_list.append(user_data)
            continue

        if len(user_data) < 6:
            train_data, test_data = train_test_split(
                user_data,
                test_size=0.2,
                random_state=random_state
            )

            train_list.append(train_data)
            test_list.append(test_data)
            continue

        train_data, temp_data = train_test_split(
            user_data,
            test_size=(1 - train_ratio),
            random_state=random_state
        )

        val_data, test_data = train_test_split(
            temp_data,
            test_size=test_ratio/(test_ratio + val_ratio),
            random_state=random_state
        )

        train_list.append(train_data)
        val_list.append(val_data)
        test_list.append(test_data)

    train_df = pd.concat(train_list).reset_index(drop=True)
    val_df = pd.concat(val_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)

    return train_df, val_df, test_df

In [ ]:
train_df, val_df, test_df = split_ratings_per_user(ratings)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

In [ ]:
def precision_at_k(recommended_items, relevant_items, k=10):
    """
    Calculate Precision@K.

    Parameters:
        recommended_items (list): Recommended movie IDs.
        relevant_items (set): Ground truth relevant movie IDs.
        k (int): Number of top recommendations.

    Returns:
        float: Precision@K score.
    """

    recommended_items = recommended_items[:k]

    hits = len(set(recommended_items) & set(relevant_items))

    return hits / k if k > 0 else 0

In [ ]:
def recall_at_k(recommended_items, relevant_items, k=10):
    """
    Calculate Recall@K.
    """

    recommended_items = recommended_items[:k]

    hits = len(set(recommended_items) & set(relevant_items))

    if len(relevant_items) == 0:
        return 0

    return hits / len(relevant_items)

In [ ]:
def average_precision(recommended_items, relevant_items, k=10):
    """
    Calculate Average Precision@K.
    """

    recommended_items = recommended_items[:k]

    score = 0.0
    hits = 0

    for i, movie in enumerate(recommended_items, start=1):

        if movie in relevant_items:
            hits += 1
            score += hits / i

    if len(relevant_items) == 0:
        return 0

    return score / min(len(relevant_items), k)

In [ ]:
def mean_average_precision(all_recommendations,
                           ground_truth,
                           k=10):

    scores = []

    for user in all_recommendations.keys():

        ap = average_precision(
            all_recommendations[user],
            ground_truth[user],
            k
        )

        scores.append(ap)

    return np.mean(scores)

In [ ]:
from sklearn.metrics import ndcg_score
def ndcg_at_k(recommended_items,
              relevant_items,
              k=10):

    recommended_items = recommended_items[:k]

    y_true = [
        1 if movie in relevant_items else 0
        for movie in recommended_items
    ]

    y_score = list(range(k, 0, -1))

    return ndcg_score([y_true], [y_score])

In [ ]:
def coverage(all_recommendations,
             total_movies):

    recommended_movies = set()

    for movies in all_recommendations.values():
        recommended_movies.update(movies)

    return len(recommended_movies) / total_movies

In [ ]:
import time

def inference_latency(recommendation_function,
                      *args,
                      **kwargs):

    start = time.time()

    recommendation_function(*args, **kwargs)

    end = time.time()

    return end - start

In [ ]:
def evaluate_recommender(
    train_df,
    test_df,
    movies,
    ratings,
    top_k=10
):
    """
    Evaluate CineAI recommender.
    """

    precision_scores = []
    recall_scores = []
    map_scores = []
    ndcg_scores = []

    all_recommendations = {}
    ground_truth = {}

    users = test_df["userId"].unique()

    for user_id in users:

        try:

            # Generate recommendations using ONLY train data
            recommendations = recommend_movies(
                user_id=user_id,
                movies=movies,
                ratings=train_df,
                top_n=top_k
            )

            if recommendations is None or len(recommendations) == 0:
                continue

            recommended_items = recommendations["movieId"].tolist()

            relevant_items = test_df[
                test_df["userId"] == user_id
            ]["movieId"].tolist()

            all_recommendations[user_id] = recommended_items
            ground_truth[user_id] = relevant_items

            precision_scores.append(
                precision_at_k(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

            recall_scores.append(
                recall_at_k(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

            map_scores.append(
                average_precision(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

            ndcg_scores.append(
                ndcg_at_k(
                    recommended_items,
                    relevant_items,
                    top_k
                )
            )

        except Exception as e:
            print(f"Skipping User {user_id}: {e}")

    coverage_score = coverage(
        all_recommendations,
        movies["movieId"].nunique()
    )

    diversity_score = diversity(
        all_recommendations,
        movies
    )

    novelty_score = novelty(
        all_recommendations,
        train_df
    )

    latency = inference_latency(
        recommend_movies,
        user_id=users[0],
        movies=movies,
        ratings=train_df,
        top_n=top_k
    )

    print("=" * 45)
    print("         CineAI Evaluation Report")
    print("=" * 45)

    print(f"Users Evaluated : {len(all_recommendations)}")

    print(f"\nPrecision@{top_k} : {np.mean(precision_scores):.4f}")
    print(f"Recall@{top_k}    : {np.mean(recall_scores):.4f}")
    print(f"MAP@{top_k}       : {np.mean(map_scores):.4f}")
    print(f"NDCG@{top_k}      : {np.mean(ndcg_scores):.4f}")

    print(f"\nCoverage          : {coverage_score:.4f}")
    print(f"Diversity         : {diversity_score:.4f}")
    print(f"Novelty           : {novelty_score:.4f}")

    print(f"\nLatency (sec)     : {latency:.4f}")

    print("=" * 45)

    return {
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "map": np.mean(map_scores),
        "ndcg": np.mean(ndcg_scores),
        "coverage": coverage_score,
        "diversity": diversity_score,
        "novelty": novelty_score,
        "latency": latency,
    }